# PyTorch Experiment Tracking

In [1]:
# Set up device agnostic code
import torch
device = "mps" if torch.mps.is_available else "cpu"
device

'mps'

In [2]:
# Set Seeds
def set_seeds(seed: int=42):
    """
    Sets random seeds for torch operations

    Args:
        seed (int, optional): Random seed to set. Defaults to 42
    """
    # Set the seed for general torch operations
    torch.manual_seed(seed)

    # Set the seed for mps torch operations
    torch.mps.manual_seed(seed)

## 1. Get Data
want to get pizza, steak, sushi images

In [3]:
import os
import zipfile
import requests

from pathlib import Path

def download_data(source: str,
                  destination: str,
                  remove_source: bool = True) -> Path:
    """Downloads a zipped dataset form source and unzips it to destination."""
    # Setup path to datafolder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder does not exist, create it
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory already exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)

        # Download the target data
        target_file = Path(source).name
        with open(data_path / target_file, "wb") as f:
            request = requests.get(source)
            print(f"[INFO] Downloading {target_file} from {source}...")
            f.write(request.content)
        
        # Unzip target file
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...")
            zip_ref.extractall(image_path)
        
        # Remove .zip_file if needed
        if remove_source:
            os.remove(data_path / target_file)
        
    return image_path


image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/refs/heads/main/data/pizza_steak_sushi.zip",
              destination="pizza_steak_sushi")

image_path

[INFO] data/pizza_steak_sushi directory already exists, skipping download.


PosixPath('data/pizza_steak_sushi')

### 2. Create Datasets and DataLoaders

In [4]:
# Set up directories
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

(PosixPath('data/pizza_steak_sushi/train'),
 PosixPath('data/pizza_steak_sushi/test'))

### 2.1 Create DataLoaders with Manual Transforms
The goal of the transforms is to ensure the your custom data is formatted in a reproducible way as well as a way that will suit pre-trained models.

In [5]:
# Manual Transform
from torchvision import transforms
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])
print(f"Manually created transforms: {manual_transforms}")

from going_modular import data_setup
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
                                                            train_dir=train_dir,
                                                            test_dir=test_dir,
                                                            transform=manual_transforms,
                                                            batch_size=32
                                                            )
train_dataloader, test_dataloader, class_names

Manually created transforms: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


(<torch.utils.data.dataloader.DataLoader at 0x12f711e90>,
 ['pizza', 'steak', 'sushi'])

### 2.2 Create DataLoaders using Auto Transform

In [6]:
import torchvision
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
auto_transform = weights.transforms()

print(f"Created Automatic Transform: {auto_transform}")

from going_modular import data_setup
train_auto_dataloader, test_auto_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=auto_transform,
    batch_size=32
)
train_auto_dataloader, test_auto_dataloader, class_names

Created Automatic Transform: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)


(<torch.utils.data.dataloader.DataLoader at 0x12f725390>,
 ['pizza', 'steak', 'sushi'])

### 3. Getting a Pre-Trained model for Transfer Learning

In [7]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model = torchvision.models.efficientnet_b0(weights=weights).to(device)
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [8]:
# Freeze all the base layers of the EffNet B0
for params in model.features.parameters():
    params.requires_grad = False

In [9]:
# Adjust the classifier head
import torch
from torch import nn
set_seeds()
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=len(class_names))
).to(device)

In [10]:
# Let us get model info using torchinfo
from torchinfo import summary

summary(
    model=model,
    input_size=[32, 3, 224, 224],
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 

### Training the model

In [11]:
# Set up loss function and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.classifier.parameters(), lr=0.001)

### Tracking Experiments

To track experiments we can use tensorboard, and to interact with tensor board we can use TensorBoard's SummaryWriter

In [12]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter()
writer

In [13]:
from tqdm.auto import tqdm 
from typing import Dict, List, Tuple

from going_modular.engine import train_step, test_step

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          epochs: int,
          device: torch.device) -> Dict[str, List]:
    
    """Trains and Tests a PyTorch Model"""

    # 1. Make an empty results dict containing:
        # training loss, training accuracy and testing loss and testing accuracy
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    # 2. Put the model on device
    model.to(device)

    # 3. start the training loop
    for epoch in tqdm(range(epochs)):
        # 4.1 Calculate train step
        train_loss, train_acc = train_step(model=model,
                                           data_loader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)
        # 4.2 Calculate test step
        test_loss, test_acc = test_step(model=model,
                                        data_loader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)
    
        # 5. print the results
        print(
            f"Epoch: {epoch+1} | " 
            f"train_loss: {train_loss} | "
            f"train_acc: {train_acc} | "
            f"test_loss: {test_loss} | "
            f"test_acc: {test_acc} | "
        )

        # 6. Add the results to the results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        # 7. Experiment Tracking
        writer.add_scalars(main_tag="Loss",
                           tag_scalar_dict={"train_loss": train_loss,
                                            "test_loss": test_loss},
                            global_step=epoch)
        
        writer.add_scalars(main_tag="Accuracy",
                           tag_scalar_dict={"train_acc": train_acc,
                                            "test_acc": test_acc},
                            global_step=epoch)
        
        writer.add_graph(model=model,
                         input_to_model=torch.randn(32, 3, 224, 224).to(device))
        
    writer.close()
    
    
    # Return the filled results at the end of the epochs
    return results

/opt/homebrew/Caskroom/miniconda/base/envs/nlp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# Train Model
# Note: not using engine.train(), since we updated the train() function above
set_seeds()
results = train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                loss_fn=loss_fn,
                optimizer=optimizer,
                epochs=5,
                device=device)

  0%|          | 0/5 [00:00<?, ?it/s]/opt/homebrew/Caskroom/miniconda/base/envs/nlp_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch: 1 | train_loss: 1.0833975076675415 | train_acc: 0.390625 | test_loss: 0.8833740949630737 | test_acc: 0.6079545454545454 | 


 20%|██        | 1/5 [00:59<03:58, 59.57s/it]

Epoch: 2 | train_loss: 0.851087249815464 | train_acc: 0.765625 | test_loss: 0.780199925104777 | test_acc: 0.7215909090909092 | 


 40%|████      | 2/5 [01:58<02:57, 59.07s/it]

...

 60%|██████    | 3/5 [02:57<01:57, 58.92s/it]

Epoch: 4 | train_loss: 0.7314413040876389 | train_acc: 0.76171875 | test_loss: 0.6146107117335001 | test_acc: 0.8958333333333334 | 


 80%|████████  | 4/5 [03:55<00:58, 58.87s/it]

Epoch: 5 | train_loss: 0.6286097094416618 | train_acc: 0.85546875 | test_loss: 0.5702572266260783 | test_acc: 0.8863636363636364 | 


100%|██████████| 5/5 [04:54<00:00, 58.90s/it]


In [15]:
results

{'train_loss': [1.0833975076675415,
  0.851087249815464,
  0.8113340139389038,
  0.7314413040876389,
  0.6286097094416618],
 'train_acc': [0.390625, 0.765625, 0.65234375, 0.76171875, 0.85546875],
 'test_loss': [0.8833740949630737,
  0.780199925104777,
  0.7157273292541504,
  0.6146107117335001,
  0.5702572266260783],
 'test_acc': [0.6079545454545454,
  0.7215909090909092,
  0.774621212121212,
  0.8958333333333334,
  0.8863636363636364]}

### 5. View our model's results with TensorBoard

In [16]:
%load_ext tensorboard
%tensorboard --logdir runs

### 6. Create a function to prepare a `SummaryWriter()` instance

By default our `SummaryWriter()` class saves to `log_dir`.

Saving different experiments to different folders is a much better option
one experiment = one folder

SummaryWriter class to include
* Experiment date/timestamp
* Experiment name
* Model name 
* Extra - Anything else that we can track

Tracking experiments to a directory in the format:

`runs/YYYY-MM-DD/experiment_name/model_name/extra`

In [17]:
from torch.utils.tensorboard import SummaryWriter
def create_writer(experiment_name: str,
                   model_name: str,
                   extra: str = None
                   ):
    """Creates a torch.utils.tensorboard.writer.SummaryWriter() instance tracking to a specific folder"""
    from datetime import datetime
    import os

    # Get timestamp of a current date in reverse order
    timestamp = datetime.now().strftime("%Y-%m-%d")

    if extra:
        # Create log directory path
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)
    
    print(f"[INFO] Created SummaryWriter saving to {log_dir}")
    
    return SummaryWriter(log_dir=log_dir)

In [18]:
example_writer = create_writer(experiment_name="data_10_percent",
              model_name="effnetb0",
              extra="5_epochs")

example_writer

[INFO] Created SummaryWriter saving to runs/2026-05-23/data_10_percent/effnetb0/5_epochs


### 6.1 Update the `train()` function to include a `writer` parameter

In [49]:
from tqdm.auto import tqdm 
from typing import List, Dict, Tuple

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          epochs: int,
          device: torch.device,
          writer: torch.utils.tensorboard.writer.SummaryWriter) -> Dict[str, List]:
    """Trains and Test a PyTorch Model"""
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }
    
    model.to(device)

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           data_loader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)
        test_loss, test_acc = test_step(model=model,
                                        data_loader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)
        
        print(f"Epoch: {epoch+1} | "
              f"train loss: {train_loss} | "
              f"train acc: {train_acc} | "
              f"test loss: {test_loss} | "
              f"test acc: {test_acc}")
        
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        if writer:
            writer.add_scalars(main_tag="Loss",
                               tag_scalar_dict={"train_loss": train_loss,
                                                "test_loss": test_loss},
                                global_step=epoch)
            
            writer.add_scalars(main_tag="Accuracy",
                               tag_scalar_dict={"train_acc": train_acc,
                                                "test_acc": test_acc},
                                global_step=epoch)
            
            writer.add_graph(model=model,
                             input_to_model=torch.randn(32, 3, 224, 224).to(device))
            
            writer.close()
        
        else:
            pass
    
    return results


### 7. Setting up a series of modelling experiments
* Setup 2x modelling experiments with effnetb0, pizza, steak, sushi data and train one model for 5 epochs and another model for 10 epochs

### 7.1 What kind of experiments should you run?

The number of machine learning experiments you can run, is like the number of different models you can build... almost limitless

However we cannot test everything, so what should we test?
* Change the number of epochs
* Change the number of hidden layers/units
* Change the amount of data (right now we're using 10% of the Food101 dataset for pizza, steak and sushi)
* Change the learning rate
* Try different kinds of data augmentation
* Choose different model architecture

This is why transfer learning is so powerful, because it's a working model that we can apply to our own problem

### 7.2 What experiments are we going to run in this notebook?

1. Model Size - EffnetB0 vs EffnetB2
2. Dataset Size - 10% of pizza, steak and sushi images vs 20% of pizza, steak and sushi images
3. Training time - 5 epochs vs 10 epochs

our goal: a model that is well performing but still small enough to run on a mobile device or web browser, so FoodVision Mini can come to life.

### 7.3 Download Different Datasets

we want two datasets:
1. Pizza, Steak, Sushi 10%
2. Pizza, Steak, Sushi 20%

In [50]:
data_10_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                                     destination="pizza_steak_sushi")

data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

[INFO] data/pizza_steak_sushi directory already exists, skipping download.
[INFO] data/pizza_steak_sushi_20_percent directory already exists, skipping download.


In [51]:
train_dir_10_percent = data_10_percent_path / "train"
train_dir_20_percent = data_20_percent_path / "train"

test_dir = data_10_percent_path / "test"

In [52]:
from torchvision import transforms

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

simple_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

In [53]:
from going_modular.data_setup import create_dataloaders
BATCH_SIZE = 32

train_dataloader_10_percent, test_dataloader, class_names = create_dataloaders(
    train_dir=train_dir_10_percent,
    test_dir=test_dir,
    transform=simple_transform,
    batch_size=BATCH_SIZE
)

train_dataloader_20_percent, test_dataloader, class_names = create_dataloaders(
    train_dir=train_dir_20_percent,
    test_dir=test_dir,
    transform=simple_transform,
    batch_size=BATCH_SIZE
)

print(f"Number of batches of size {BATCH_SIZE} in 10 percent training data: {len(train_dataloader_10_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in 20 percent training data: {len(train_dataloader_20_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in testing data: {len(test_dataloader)} (all experiments will use the same test set)")
print(f"Number of classes: {len(class_names)}, class names: {class_names}")

Number of batches of size 32 in 10 percent training data: 8
Number of batches of size 32 in 20 percent training data: 15
Number of batches of size 32 in testing data: 3 (all experiments will use the same test set)
Number of classes: 3, class names: ['pizza', 'steak', 'sushi']


### 7.5 Create feature extractor models

We want two functions

1. Creates a `torchvision.models.efficientnet_b0()` feature extractor with a frozen backbone/base layers and a custom classifier head (EffNetB0)
2. Creates a `torchvision.models.efficientnet_b2()` feature extractor with a frozen backbone/base layers and a custom classifier head (EffNetB2)

In [54]:
import torchvision

# Create an EffNetB2
effnetb2_weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
effnetb2 = torchvision.models.efficientnet_b2(weights=effnetb2_weights)



In [55]:
effnetb2

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [56]:
from torchinfo import summary

summary(
    model=effnetb2,
    input_size=[32, 3, 224, 224],
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 1000]           --                   True
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1408, 7, 7]     --                   True
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   True
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   864                  True
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   64                   True
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 16, 112

In [57]:
import torchvision
from torch import nn

OUT_FEATURES = len(class_names)

# Create an EffNetB0 feature extractor
def create_effnetb0():
    # Get the weights and setup a model
    weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
    model = torchvision.models.efficientnet_b0(weights=weights).to(device)

    # Freeze the base model layers
    for param in model.parameters():
        param.requires_grad = False
    
    # Change the classifier head
    set_seeds()
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features=1280, out_features=OUT_FEATURES)
    ).to(device)

    # Give the model a name
    model.name = "effnetb0"
    print(f"[INFO] Created new {model.name} model...")
    return model

def create_effnetb2():
    effnetb2_weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    model = torchvision.models.efficientnet_b2(weights=effnetb2_weights).to(device)

    for params in model.features.parameters():
        params.requires_grad = False
    
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features=1408, out_features=OUT_FEATURES)
    ).to(device)

    model.name = "effnetb2"
    print(f"[INFO] Created new {model.name} model...")
    return model

In [58]:
created_test_model = create_effnetb0()
summary(
    model=created_test_model,
    input_size=[32, 3, 224, 224],
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

[INFO] Created new effnetb0 model...


Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 

In [59]:
created_test_model_2 = create_effnetb2()
summary(
    model=created_test_model_2,
    input_size=[32, 3, 224, 224],
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

[INFO] Created new effnetb2 model...


Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1408, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 

### 7.6 Create Experiments and set up training code

In [60]:
# Create epoch list
num_epochs = [5, 10]

# Create models list (need to create a new model for each experiment)
models = ["effnetb0", "effnetb2"]

# Create a DataLoaders dictionary
train_dataloaders = {
    "data_10_percennt": train_dataloader_10_percent,
    "data_20_percent": train_dataloader_20_percent
}

In [ ]:
# %%time
# from going_modular.utils import save_model

# # Set Seeds
# set_seeds(seed=42)

# # Keep track of experiments numbers
# experiment_number = 0

# # Loop through each DataLoader
# for dataloader_name, train_dataloader in train_dataloaders.items():
#     # Loop through the epochs
#     for epochs in num_epochs:
#         # Loop through each model name and create a new model instance
#         for model_name in models:
#             experiment_number += 1

#             # Print out info
#             print(f"[INFO] Experiment number: {experiment_number}")
#             print(f"[INFO] Model: {model_name}")
#             print(f"[INFO] DataLoader: {dataloader_name}")
#             print(f"[INFO] Number of Epochs: {epochs}")

#             # Select and create the model
#             if model_name == "effnetb0":
#                 model = create_effnetb0()
#             else:
#                 model = create_effnetb2()
            
#             # Create a new loss and optimizer for every model
#             loss_fn = nn.CrossEntropyLoss()
#             optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

#             # Train target model with target dataloader and track experiments
#             train(model=model,
#                   train_dataloader=train_dataloader,
#                   test_dataloader=test_dataloader,
#                   optimizer=optimizer,
#                   loss_fn=loss_fn,
#                   epochs=epochs,
#                   device=device,
#                   writer=create_writer(experiment_name=dataloader_name,
#                                        model_name=model_name,
#                                        extra=f"{epochs}_epochs"))
            
#             # Save the model to file so we can import it later if needed be
#             save_filepath = f"07_{model_name}_{dataloader_name}_{epochs}_epochs.pth"
#             save_model(model=model,
#                        target_dir="models",
#                        model_name=save_filepath)
#             print("-"*50 + "\n")

[INFO] Experiment number: 1
[INFO] Model: effnetb0
[INFO] DataLoader: data_10_percennt
[INFO] Number of Epochs: 5
[INFO] Created new effnetb0 model...
[INFO] Created SummaryWriter saving to runs/2026-05-23/data_10_percennt/effnetb0/5_epochs


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | train loss: 1.062392957508564 | train acc: 0.46484375 | test loss: 0.8881809711456299 | test acc: 0.5568181818181818


 20%|██        | 1/5 [00:58<03:54, 58.74s/it]

Epoch: 2 | train loss: 0.9345129653811455 | train acc: 0.546875 | test loss: 0.7581113974253336 | test acc: 0.8030303030303031


...

Epoch: 9 | train loss: 0.36227121353149416 | train acc: 0.8875 | test loss: 0.4317738115787506 | test acc: 0.8977272727272728


 90%|█████████ | 9/10 [13:27<01:29, 89.68s/it]

Epoch: 10 | train loss: 0.3490408450365067 | train acc: 0.8875 | test loss: 0.4067018727461497 | test acc: 0.9280303030303031


100%|██████████| 10/10 [14:57<00:00, 89.71s/it]

[INFO] Saving model to: models/07_effnetb2_data_20_percent_10_epochs.pth
--------------------------------------------------

CPU times: user 1min 15s, sys: 48.2 s, total: 2min 3s
Wall time: 1h 14min 19s
